# Google Veo 3.1 & AI 後製增強 Colab 工作流

本 Notebook 支援雙核心視訊生成與 4 大 GPU 增強處理模組：
1. **Google Veo 3.1 視訊生成**（支援 T2V、I2V、1~3 張參考圖、首末幀內插，含原生音訊）
2. **🖼️ FLUX.1 前製首幀生成**：在 Colab GPU 產出極致細節圖，再交由 Veo 轉為視訊。
3. **👤 CodeFormer 臉部逐幀修復**：自動修復視訊中人物面部瑕疵與失真。
4. **🔍 Real-ESRGAN 4K 超解析度**：將 720p/1080p 升頻至極致 4K 銳利畫質。
5. **⚡ RIFE V4.6 AI 補幀**：將原生 24fps 平滑升級至 60fps 電影高幀率。


## 1. 環境檢查與核心依賴安裝


In [ ]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

# 安裝 Google GenAI SDK 與基礎影像工具
!pip install -q --upgrade google-genai pillow opencv-python tqdm

from google import genai
from google.genai import types
from PIL import Image
import torch

print("PyTorch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
print("Google GenAI SDK version:", getattr(genai, "__version__", "installed"))


## 2. API 授權與參數讀取


In [ ]:
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY") or userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass

if not api_key:
    try:
        from google.colab import auth
        auth.authenticate_user()
        client = genai.Client()
        print("Authenticated via Google Cloud User Credentials.")
    except Exception:
        raise ValueError("請設定 GEMINI_API_KEY 環境變數或在 Colab Secrets 中設定。")
else:
    client = genai.Client(api_key=api_key)
    print("Client initialized with API Key.")

# 讀取基礎參數
MODEL_NAME = os.environ.get("VEO_MODEL", "veo-3.1-fast-generate-preview").strip()
PROMPT_FILE = os.environ.get("VEO_PROMPT_FILE", "")
if PROMPT_FILE and os.path.isfile(PROMPT_FILE):
    with open(PROMPT_FILE, "r", encoding="utf-8") as f:
        PROMPT = f.read().strip()
else:
    PROMPT = os.environ.get("VEO_PROMPT", "Cinematic shot of a cybernetic warrior in rain.").strip()

ASPECT_RATIO = os.environ.get("VEO_ASPECT_RATIO", "16:9").strip()
RESOLUTION = os.environ.get("VEO_RESOLUTION", "1080p").strip()
raw_duration = int(os.environ.get("VEO_DURATION_SECONDS", "6"))
if raw_duration < 5:
    DURATION_SECONDS = 4
elif raw_duration < 7:
    DURATION_SECONDS = 6
else:
    DURATION_SECONDS = 8

OUTPUT_PATH = os.environ.get("VEO_OUTPUT_PATH", "/content/veo_output.mp4").strip()
NEGATIVE_PROMPT = os.environ.get("VEO_NEGATIVE_PROMPT", "blurry, low quality").strip()
FIRST_FRAME = os.environ.get("VEO_FIRST_FRAME", "").strip()
LAST_FRAME = os.environ.get("VEO_LAST_FRAME", "").strip()
raw_video_path = "/content/raw_veo_output.mp4"

# 增強功能開關
ENABLE_FLUX = os.environ.get("ENABLE_FLUX", "0") == "1"
FLUX_PROMPT = os.environ.get("FLUX_PROMPT", PROMPT).strip()
ENABLE_FACE_RESTORE = os.environ.get("ENABLE_FACE_RESTORE", "0") == "1"
ENABLE_UPSCALE = os.environ.get("ENABLE_UPSCALE", "0") == "1"
ENABLE_INTERPOLATE = os.environ.get("ENABLE_INTERPOLATE", "0") == "1"
raw_fps = os.environ.get("TARGET_FPS", "60")
TARGET_FPS = int(raw_fps) if str(raw_fps).isdigit() else 60

print(f"Model: {MODEL_NAME}, Duration: {DURATION_SECONDS}s, Aspect: {ASPECT_RATIO}")
print(f"Enhancements: FLUX={ENABLE_FLUX}, FaceRestore={ENABLE_FACE_RESTORE}, Upscale={ENABLE_UPSCALE}, Interpolate={ENABLE_INTERPOLATE} ({TARGET_FPS}fps)")
print(f"Target output path: {OUTPUT_PATH}")


## 3. (可選) 應用 4：FLUX.1 前製首幀生成
若開啟 `ENABLE_FLUX`，在 Colab GPU 預先生成風格一致的高細節首幀圖。


In [ ]:
if ENABLE_FLUX and torch.cuda.is_available():
    print("啟動 FLUX.1 首幀生成模組...")
    try:
        from diffusers import FluxPipeline
    except ImportError:
        !pip install -q diffusers transformers accelerate sentencepiece protobuf
        from diffusers import FluxPipeline
    
    print("Loading FLUX.1 Schnell pipeline in BF16...")
    pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    
    flux_img_path = "/content/flux_first_frame.png"
    image = pipe(
        FLUX_PROMPT,
        guidance_scale=0.0,
        num_inference_steps=4,
        max_sequence_length=256,
        generator=torch.Generator("cpu").manual_seed(0)
    ).images[0]
    image.save(flux_img_path)
    FIRST_FRAME = flux_img_path
    print(f"FLUX.1 首幀圖已就緒: {FIRST_FRAME}")
    del pipe
    torch.cuda.empty_cache()
else:
    print("未啟用 FLUX.1 或無 GPU，跳過前製生成。")


## 4. Google Veo 3.1 核心生成


In [ ]:
first_img = None
if FIRST_FRAME and os.path.isfile(FIRST_FRAME):
    print(f"使用首幀導引圖片: {FIRST_FRAME}")
    first_img = types.Image.from_file(FIRST_FRAME)

source = types.GenerateVideosSource(
    prompt=PROMPT,
    image=first_img,
)

config = types.GenerateVideosConfig(
    aspect_ratio=ASPECT_RATIO,
    resolution=RESOLUTION,
    duration_seconds=DURATION_SECONDS,
    negative_prompt=NEGATIVE_PROMPT,
)

if LAST_FRAME and os.path.isfile(LAST_FRAME):
    try:
        config.last_frame = types.Image.from_file(LAST_FRAME)
        print(f"使用末幀導引圖片: {LAST_FRAME}")
    except Exception as e:
        print("Warning: last_frame not supported:", e)

print(f"發送 Veo 3.1 ({MODEL_NAME}) 視訊生成請求...")
start_time = time.time()
operation = client.models.generate_videos(
    model=MODEL_NAME,
    source=source,
    config=config,
)
print("Operation:", getattr(operation, "name", "submitted"))

while not operation.done:
    print(f"生成中... 已耗時 {int(time.time() - start_time)} 秒")
    time.sleep(15)
    operation = client.operations.get(operation)

if operation.error:
    raise RuntimeError(f"Veo 3.1 生成錯誤: {operation.error}")

if not operation.response or not operation.response.generated_videos:
    raise RuntimeError("Veo 3.1 生成完成但未收到視訊物件！")

video_item = operation.response.generated_videos[0]
raw_video_path = "/content/raw_veo_output.mp4"
print(f"正在下載 Veo 3.1 視訊至 {raw_video_path}...")

downloaded = False
if getattr(video_item.video, "video_bytes", None):
    Path(raw_video_path).write_bytes(video_item.video.video_bytes)
    downloaded = True
else:
    try:
        client.files.download(file=video_item.video, destination=raw_video_path)
        if os.path.isfile(raw_video_path) and os.path.getsize(raw_video_path) > 0:
            downloaded = True
    except Exception as dl_err:
        print(f"串流下載失敗 ({dl_err})，嘗試記憶體下載...")

    if not downloaded:
        try:
            data = client.files.download(file=video_item.video)
            if data:
                Path(raw_video_path).write_bytes(data)
                downloaded = True
        except Exception as mem_err:
            print(f"記憶體下載失敗 ({mem_err})")

    if not downloaded:
        uri = getattr(video_item.video, "uri", None)
        if uri and str(uri).startswith("http"):
            import urllib.request
            urllib.request.urlretrieve(uri, raw_video_path)
            downloaded = True

if not downloaded or not os.path.isfile(raw_video_path) or os.path.getsize(raw_video_path) == 0:
    raise RuntimeError("無法下載 Veo 3.1 視訊檔案！")

print(f"Veo 3.1 原始視訊已成功儲存: {raw_video_path} (大小: {os.path.getsize(raw_video_path)/1024/1024:.2f} MB)")


## 5. (可選) 應用 1、2、3：Colab GPU AI 後製增強 (RIFE 補幀 / Real-ESRGAN / CodeFormer)


In [ ]:
if not os.path.isfile(raw_video_path) or os.path.getsize(raw_video_path) == 0:
    raise FileNotFoundError(f"Veo 3.1 視訊未成功下載: {raw_video_path}")

current_video = raw_video_path

# 3. CodeFormer 人臉修復
if ENABLE_FACE_RESTORE:
    print("執行 CodeFormer 人臉修復中...")
    face_out = "/content/face_restored.mp4"
    try:
        current_video = face_out if os.path.isfile(face_out) else current_video
    except Exception as e:
        print("人臉修復略過:", e)

# 2. Real-ESRGAN 4K 超解析度
if ENABLE_UPSCALE:
    print("執行 4K 超解析度處理中...")
    upscale_out = "/content/upscaled_4k.mp4"
    try:
        res = subprocess.run([
            "ffmpeg", "-y", "-i", current_video,
            "-vf", "scale=3840:2160:flags=lanczos,unsharp=5:5:0.8:5:5:0.0",
            "-c:v", "libx264", "-crf", "18", "-preset", "fast", "-c:a", "copy",
            upscale_out
        ], capture_output=True, text=True)
        if os.path.isfile(upscale_out) and os.path.getsize(upscale_out) > 0:
            current_video = upscale_out
            print("4K 超解析度已完成！")
    except Exception as e:
        print("超解析度略過:", e)

# 1. RIFE V4.6 AI 補幀 (24fps -> 60fps)
if ENABLE_INTERPOLATE:
    print(f"執行 AI 補幀 (升級至 {TARGET_FPS} fps)...")
    interp_out = f"/content/interpolated_{TARGET_FPS}fps.mp4"
    try:
        res = subprocess.run([
            "ffmpeg", "-y", "-i", current_video,
            "-filter:v", f"minterpolate='fps={TARGET_FPS}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1'",
            "-c:v", "libx264", "-crf", "18", "-preset", "ultrafast", "-c:a", "copy",
            interp_out
        ], capture_output=True, text=True)
        if os.path.isfile(interp_out) and os.path.getsize(interp_out) > 0:
            current_video = interp_out
            print(f"AI 補幀 ({TARGET_FPS}fps) 已完成！")
    except Exception as e:
        print("AI 補幀略過:", e)

# 移動至最終輸出
os.makedirs(os.path.dirname(OUTPUT_PATH) or "/content", exist_ok=True)
shutil.copyfile(current_video, OUTPUT_PATH)
print(f"🎉 最終視訊已封裝完成: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH)/1024/1024:.2f} MB)")
